In [1]:
# Setup ACT paths using path_config
import os
from pathlib import Path
import sys

import torch

act_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if act_root not in sys.path:
    sys.path.insert(0, act_root)

ACT_ROOT = Path(act_root)

In [2]:
from act.util.device_manager import initialize_device


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
initialize_device(device=DEVICE, dtype="float32")

✅ Device Manager Initialized: device=cpu, dtype=torch.float32


In [3]:
from act.front_end.model_synthesis import synthesize_models_from_specs
from act.front_end.torchvision_loader.create_specs import TorchVisionSpecCreator


spec_creator = TorchVisionSpecCreator()

specs = spec_creator.create_specs_for_data_model_pairs(
    dataset_names=["CUB200"],
    model_names=["resnet18"],
)

synthesis_models = synthesize_models_from_specs(specs)

[ACT] Auto-detecting project root: ..
[ACT] Gurobi license found: ../modules/gurobi/gurobi.lic

LOADING: CUB200 + resnet18
[1/3] Loading dataset (test split)...
  ✓ Loaded 5794 samples
[2/3] Loading model architecture...
  ✓ Loaded resnet18 with pre-trained weights
  ✓ Adjusted final layer: 512 → 200 classes
  ✓ Loaded resnet18 from torchvision.models
[3/3] Summary...
  Dataset: 5794 samples (test split)
  Model: 11,279,112 parameters (11,279,112 trainable)
  Batch size: 1
  Preprocessing: Yes

✓ LOADED SUCCESSFULLY

🧬 Synthesizing models from 1 spec result(s)...

🎉 Synthesis Complete:
   Total specs: 240
   Wrapped models: 4


In [4]:
# ACT fuzzing config
from act.front_end.spec_creator_base import LabeledInputTensor
from act.front_end.specs import InKind
from act.front_end.verifiable_model import InputLayer, InputSpecLayer, VerifiableModel
from act.pipeline.fuzzing.actfuzzer import ACTFuzzer, FuzzingConfig


config_path = ACT_ROOT / "act/pipeline/fuzzing/config.yaml"
config = FuzzingConfig.from_yaml(
    config_path=config_path,
    max_iterations=500,
    timeout_seconds=60.0,
    report_interval=20,
    verbose=1,
    # Tracing: capture execution traces for analysis
    trace_level=1,
    trace_storage='json')

all_reports = []
all_fuzzers = []

# Metadata for instances
instance_info = []

# Iterate over wrapped models for fuzzing results
for idx, item in enumerate(synthesis_models.items()):
    model: VerifiableModel = item[1] # type: ignore

    input_layer: InputLayer = model.input_layer
    labeled_input = input_layer.labeled_input

    input_spec_layer: InputSpecLayer = model.input_spec
    input_spec = input_spec_layer.spec

    images = labeled_input.tensor.to(DEVICE)
    labels = labeled_input.label.to(DEVICE)

    # ACT now combines models, thus we need to infer the new batch size
    # of that synthesised model and change seeds (which expect only one class label)
    # accordingly
    batch_size = images.shape[0]

    initial_seeds = [
        LabeledInputTensor(tensor=images[i:i+1], label=labels[i:i+1])
        for i in range(batch_size)
    ]

    if input_spec_layer.kind == InKind.BOX:
        epsilon = float((input_spec.ub - input_spec.lb).max())
    else:
        # L_INF
        epsilon = input_spec_layer.eps

    # TODO: find ways to improve coverage of fuzzer after iterations
    # (possibility that a lot of the uncovered neurons are irrelevant to class prediction)
    fuzzer = ACTFuzzer(
        wrapped_model=model,
        initial_seeds=initial_seeds,
        config=config
    )

    # Store instance info for visualization
    instance_info.append({
        'index': idx,
        'input_tensor': labeled_input.tensor,
        'true_label': labeled_input.label,
        'epsilon': epsilon,
        'input_spec': input_spec
    })
    
    report = fuzzer.fuzz()

    all_reports.append(report)
    all_fuzzers.append(fuzzer)

INFO:numexpr.utils:NumExpr defaulting to 4 threads.


[MutationEngine] Adaptive Scalar Perturbation Size:
  - perturb_scale: 0.1 (fraction of range per perturbation)
  - mean_range: 0.093088
  - computed perturb_size: 0.009309
  - steps_to_traverse: ~10.0 steps
  - interpretation: Each mutation perturbation covers 10.0% of the range
📊 Tracing enabled: Level 1, sampling every 1 iteration(s)
   Output: ../act/pipeline/log/fuzzing_results/traces_0.json
ACT: Abstract Constraint Transformer
Inference-based whitebox fuzzing for neural network verification

🚀 Starting ACTFuzzer with 10 seeds
   Device: cpu
   Batch size: 80 (from model synthesis)
   Max iterations: 500
   Timeout: 60.0s

📊 Iteration     80 | GlobalCov: 98.83% BestInputCov: 97.88% | Seeds:   90 | Violations:  80 (+80) | Speed:   0.5 it/s (41 samples/s)
⏱️  Timeout reached after 80 iterations

🎉 ACTFuzzer completed in 155.1s
   Iterations: 80
   Counterexamples: 80
   GlobalCov: 98.83%  BestInputCov: 97.88%
   Seeds explored: 90
   Never-activated neurons: 82
   Never-activated sa